# Method comparison: robust z-score baseline vs Isolation Forest vs DBSCAN

All methods use the same rows (`row_id`), the same 4 features and the same peer groups (min size 100).
We compare the top 1% of each method by its raw score. DBSCAN scored level 1 rows only,
so the 3-method comparison is on level 1; fallback rows (level 2-3) are compared for baseline and IF.

## 1. Load

In [1]:
import numpy as np
import pandas as pd

feats = ["log_pay", "log_benes", "srvcs_per_bene", "markup"]
rz_cols = [f + "_rz" for f in feats]
keys = ["Rndrng_NPI", "HCPCS_Cd", "Place_Of_Srvc"]

feat = pd.read_parquet("data/processed/features.parquet")  # row order = row_id
peers = pd.read_parquet("data/processed/peers_100.parquet", columns=["row_id", "peer_level"] + rz_cols)
iso = pd.read_csv("isolation_forest_full_anomaly_ranking.csv", usecols=["row_id", "iforest_raw_score"])
db = pd.read_csv("all_results.csv", usecols=keys + ["cluster", "neighbor_distance"],
                 dtype={k: str for k in keys})  # text, so codes like 00100 keep their zeros

lvl = peers["peer_level"]
t = feat["Rndrng_Prvdr_Type"]

## 2. One score per method, in row_id order

In [2]:
# baseline: 1 feature > 10 or 2+ features > 3.5  <=>  max(2nd largest z, largest z * 3.5/10) > 3.5
high = np.sort(peers[rz_cols].clip(lower=0).fillna(0).to_numpy(), axis=1)  # only above peers
base = pd.Series(np.maximum(high[:, -2], high[:, -1] * 3.5 / 10)).where(lvl > 0)

# Isolation Forest: raw score (-score_samples), bigger = more anomalous
ifs = iso.set_index("row_id")["iforest_raw_score"].sort_index().reset_index(drop=True)

# DBSCAN: no row_id in the file, so match on NPI + HCPCS + place of service
ids = feat[["row_id"] + keys].astype({k: str for k in keys})
db = db.merge(ids, on=keys)
dbs = pd.Series(np.nan, index=feat.index)
dbs[db["row_id"]] = db["neighbor_distance"].to_numpy()  # bigger = more isolated
noise = pd.Series(False, index=feat.index)
noise[db["row_id"]] = (db["cluster"] == -1).to_numpy()  # DBSCAN's own flag

## 3. Top 1% of each method

In [3]:
def top_set(score, rows, q=0.01):
    s = score.where(rows)
    return s >= s.quantile(1 - q)  # NaN is never in the top

def jaccard(a, b):
    return (a & b).sum() / (a | b).sum() * 100  # both / either

# baseline vs IF: all levels
scored2 = base.notna() & ifs.notna()
top_base, top_if = top_set(base, scored2), top_set(ifs, scored2)

# all 3 methods: rows every method scored (level 1)
scored3 = scored2 & dbs.notna()
tops = {"baseline": top_set(base, scored3), "IF": top_set(ifs, scored3), "DBSCAN": top_set(dbs, scored3)}

print(f"rows: baseline + IF {scored2.sum():,}, all three {scored3.sum():,}")
print(f"Jaccard if methods picked at random: {0.01 / 1.99 * 100:.2f}%")

rows: baseline + IF 9,704,467, all three 9,098,095
Jaccard if methods picked at random: 0.50%


## 4. Overlap

In [4]:
names = list(tops)
pd.DataFrame({a: {b: jaccard(tops[a], tops[b]) for b in names} for a in names}).round(1)  # Jaccard %, top 1%

,baseline,IF,DBSCAN
baseline,100.0,17.5,13.3
IF,17.5,100.0,19.8
DBSCAN,13.3,19.8,100.0


In [5]:
# how many methods put a record in their top 1%
n_methods = sum(tops.values())
print(n_methods[scored3].value_counts().sort_index().rename("records"))
print(f"all three: {(n_methods == 3).sum():,}  (expected at random: {scored3.sum() * 0.01 ** 3:.0f})")

0    8894013
1     144984
2      49335
3       9763
Name: records, dtype: int64
all three: 9,763  (expected at random: 9)


In [6]:
# agreement on the whole ranking, not only the top (random sample of rows)
idx = scored3[scored3].sample(500_000, random_state=0).index
pd.DataFrame({"baseline": base[idx], "IF": ifs[idx], "DBSCAN": dbs[idx]}).corr(method="spearman").round(2)

,baseline,IF,DBSCAN
baseline,1.00,0.50,0.34
IF,0.50,1.00,0.51
DBSCAN,0.34,0.51,1.00


In [7]:
# DBSCAN's own flag (noise) vs the top 1% sets
print(f"noise: {noise[scored3].mean() * 100:.2f}% of rows")
pd.Series({n: jaccard(noise & scored3, s) for n, s in tops.items()}, name="Jaccard with noise (%)").round(1)

noise: 3.90% of rows


baseline     8.9
IF          14.3
DBSCAN      25.7
Name: Jaccard with noise (%), dtype: float64

In [13]:
n = {k: v.sum() for k, v in tops.items()}
for a, b in [("baseline", "IF"), ("baseline", "DBSCAN"), ("IF", "DBSCAN")]:
    inter = (tops[a] & tops[b]).sum()
    print(a, b, "| top sizes:", n[a], n[b], "| both:", inter, "| either:", n[a] + n[b] - inter)

baseline IF | top sizes: 90981 90981 | both: 27130 | either: 154832
baseline DBSCAN | top sizes: 90981 90981 | both: 21397 | either: 160565
IF DBSCAN | top sizes: 90981 90981 | both: 30097 | either: 151865


## 5. Why baseline and IF differ

In [8]:
# by peer level: fallback rows (level 2-3) are mixed groups
pd.DataFrame({L: {"rows": (lvl == L).sum(),
                  "baseline top 1% (%)": top_base[lvl == L].mean() * 100,
                  "IF top 1% (%)": top_if[lvl == L].mean() * 100,
                  "Jaccard (%)": jaccard(top_base & (lvl == L), top_if & (lvl == L))}
              for L in [1, 2, 3]}).T.round(2)

,rows,baseline top 1% (%),IF top 1% (%),Jaccard (%)
1,9249171.0,0.90,0.93,16.01
2,426925.0,2.36,2.09,21.91
3,28371.0,12.54,6.34,17.84


In [9]:
# by provider type: % of each type's rows in the top 1% (types with 10k+ rows)
n_type = t[scored2].value_counts()
rate = pd.DataFrame({"rows": n_type,
                     "baseline (%)": t[top_base].value_counts() / n_type * 100,
                     "IF (%)": t[top_if].value_counts() / n_type * 100}).fillna(0)
rate = rate[rate["rows"] >= 10_000]
rate["baseline / IF"] = rate["baseline (%)"] / rate["IF (%)"]
display(rate.sort_values("baseline / IF", ascending=False).head(8).round(2))  # baseline favors
rate.sort_values("baseline / IF").head(8).round(2)                            # IF favors

,rows,baseline (%),IF (%),baseline / IF
Rndrng_Prvdr_Type,,,,
Pharmacy,12559,22.54,1.88,12.00
Mass Immunizer Roster Biller,157234,16.50,1.71,9.64
Centralized Flu,62761,6.69,1.80,3.71
Ambulatory Surgical Center,57620,5.69,2.17,2.63
Pathology,143431,1.15,0.58,1.97
Independent Diagnostic Testing Facility (IDTF),31408,2.28,2.03,1.12
Certified Registered Nurse Anesthetist (CRNA),115018,1.38,1.23,1.12
Geriatric Medicine,16477,1.19,1.10,1.08


,rows,baseline (%),IF (%),baseline / IF
Rndrng_Prvdr_Type,,,,
Chiropractic,41090,0.00,0.61,0.00
Physical Therapist in Private Practice,408403,0.01,0.28,0.05
Podiatry,170474,0.26,1.13,0.23
Hand Surgery,25175,0.24,1.03,0.24
Occupational Therapist in Private Practice,49485,0.11,0.46,0.24
Psychiatry,56925,0.24,0.74,0.32
Orthopedic Surgery,260150,0.37,1.11,0.34
Otolaryngology,84843,0.44,1.13,0.39


In [10]:
# IF-only records: far below peers, one moderate feature, or a combination
z = peers[rz_cols]
low_only = (z.min(axis=1) < -3.5) & (z.max(axis=1) < 3.5)  # far below peers, not above
only_if = top_if & ~top_base
mx = z.max(axis=1)
pd.Series({"below peers only (baseline ignores)": low_only[only_if].mean(),
           "one feature z 3.5-10 (baseline needs > 10)": ((mx >= 3.5) & (mx <= 10) & ~low_only)[only_if].mean(),
           "all z < 3.5 (combination)": ((mx < 3.5) & ~low_only)[only_if].mean()},
          name="% of IF-only").mul(100).round(1)

below peers only (baseline ignores)           38.3
one feature z 3.5-10 (baseline needs > 10)    50.4
all z < 3.5 (combination)                      5.6
Name: % of IF-only, dtype: float64

## 6. Sample for manual review (50 rows)

Each group answers one question. For each row: real values, peer median, how many times above the median, z-scores and the rank in each method.

In [11]:
mi = t == "Mass Immunizer Roster Biller"
B, I, D = tops["baseline"], tops["IF"], tops["DBSCAN"]

groups = {  # name: (rows, how many)
    "all three": (B & I & D, 8),
    "IF + DBSCAN, not baseline": (I & D & ~B, 5),
    "baseline only: Mass Immunizer": (B & ~I & ~D & mi, 5),
    "baseline only: other": (B & ~I & ~D & ~mi, 6),
    "IF only: below peers": (I & ~B & ~D & low_only, 5),
    "IF only: other": (I & ~B & ~D & ~low_only, 5),
    "DBSCAN only": (D & ~B & ~I, 6),
    "fallback (level 2-3)": ((top_base | top_if) & (lvl > 1), 10),  # DBSCAN didn't score level 2-3
}
taken = pd.Series(False, index=feat.index)
picked = []
for name, (mask, n) in groups.items():
    pool = mask & ~taken  # a row goes into one group only
    ids = pool[pool].sample(min(n, pool.sum()), random_state=42).index
    taken[ids] = True
    picked.append(pd.DataFrame({"row_id": ids, "review_group": name}))
s = pd.concat(picked, ignore_index=True).merge(feat, on="row_id")
s["peer_level"] = lvl[s["row_id"]].values

In [ ]:
# peer median: median of the whole group on the row's level
levels = {1: ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"], 2: ["HCPCS_Cd", "Place_Of_Srvc"], 3: ["HCPCS_Cd"]}
s = pd.concat([s[s["peer_level"] == L].merge(feat.groupby(k, observed=True)[feats].median().add_suffix("_med").reset_index(),
                                             on=k, how="left")
               for L, k in levels.items()], ignore_index=True)

labels = {"log_pay": "std. payment ($)", "log_benes": "patients",
          "srvcs_per_bene": "services per patient", "markup": "charge / allowed"}
out = s[["review_group", "row_id"] + keys + ["Rndrng_Prvdr_Type", "peer_level"]].copy()
zr = z.loc[s["row_id"]].to_numpy()
for j, (f, name) in enumerate(labels.items()):
    out[name] = np.exp(s[f]).round(2)         # features are logs, exp = real value
    out[name + " peer median"] = np.exp(s[f + "_med"]).round(2)
    out[name + " x median"] = np.exp(s[f] - s[f + "_med"]).round(2)   # 2.0 = twice the median, 0.5 = half
    out[name + " z"] = zr[:, j].round(1)
out["above peers (z > 3.5)"] = [", ".join(n for n, v in zip(labels.values(), r) if v > 3.5) for r in zr]
out["below peers (z < -3.5)"] = [", ".join(n for n, v in zip(labels.values(), r) if v < -3.5) for r in zr]

for name, score in {"baseline": base, "IF": ifs, "DBSCAN": dbs}.items():
    out[name + " pct"] = (score.rank(pct=True) * 100)[s["row_id"]].round(2).values  # 99.9 = top 0.1%; empty = not scored
out["DBSCAN noise"] = noise[s["row_id"]].values

for col in ["what stands out", "plausible explanation", "needs a closer look? (yes / no / unclear)"]:
    out[col] = ""  # for the reviewers

out = out.sort_values(["review_group", "IF pct"], ascending=[True, False])
out.to_csv("manual_review_sample.csv", index=False)
out["review_group"].value_counts()

review_group
fallback (level 2-3)             10
all three                         8
DBSCAN only                       6
baseline only: other              6
IF only: other                    5
IF only: below peers              5
IF + DBSCAN, not baseline         5
baseline only: Mass Immunizer     5
Name: count, dtype: int64